[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meiliuwu/intro-geoai/blob/main/labs/week01-model-architectures/week01_lab_same_data_four_architectures.ipynb)

# Week 1 Lab · Same data, four architectures
**[Introduction to Geospatial AI](https://github.com/meiliuwu/intro-geoai) · Putting space into neural networks**

In the lecture we asked: *what makes an AI model spatial?* Today you test the answer on one dataset, about 20,000 Californian census block groups, by giving models information about location in four different ways and measuring what each one buys you.

| Part | What you do | Time |
|---|---|---|
| 0 | Set up a reproducible notebook; load and map the data | 10 min |
| 1 | Baselines with and without coordinates | 10 min |
| 2 | Spatial autocorrelation, and random vs spatial cross-validation | 20 min |
| 3 | Neural networks (MLPs) with three kinds of location input | 25 min |
| 4 | A convolution is a GIS focal operation | 15 min |
| 5 | A graph neural network on a neighbourhood graph | 25 min |
| 6 | Compare, reflect and save | 10 min |

**By the end you can:** explain why random cross-validation flatters spatial models; build an MLP with a multi-scale location encoding; show that a CNN filter is a focal (moving-window) operation; and train a graph neural network by message passing in plain PyTorch.

**How to work.** Work in pairs: one *driver* types, one *navigator* reads ahead and checks. Swap at each Part. Cells marked **Task** ask you to change or write something; **Question** cells ask you to write a short answer in the cell below them. Solution cells are hidden; double-click to open them *after* you have tried.

**Before you start.** Choose **File → Save a copy in Drive** so your changes are kept, then work in that copy.

**Runtime.** A CPU runtime is enough; no GPU is needed this week. The whole notebook runs in a few minutes.

## Part 0 · Set up and look at the data

Reproducible GeoAI starts on line one: fix the random seeds, record library versions, and record where the data came from. You will reuse this pattern in every lab and in your project repository.

In [ ]:
# Part 0 - setup (run once)
import os, random, platform, time, warnings
warnings.filterwarnings('ignore', message='Sparse invariant checks')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import sklearn
from scipy import ndimage
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold, GroupKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
set_seed()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
VERSIONS = (f"Python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | "
            f"scikit-learn {sklearn.__version__} | torch {torch.__version__} | device: {device}")
print(VERSIONS)

**The data.** Each row is a California census block group from the 1990 US Census, the dataset Pace and Barry (1997) built to demonstrate *spatial* autoregression. The target `MedHouseVal` is the median house value in units of \$100,000 (capped at 5.0). Colab ships a copy in `sample_data/`, so nothing needs downloading; elsewhere the cell falls back to scikit-learn's copy.

In [ ]:
COLAB_DIR = '/content/sample_data'
if os.path.exists(f'{COLAB_DIR}/california_housing_train.csv'):
    raw = pd.concat([pd.read_csv(f'{COLAB_DIR}/california_housing_train.csv'),
                     pd.read_csv(f'{COLAB_DIR}/california_housing_test.csv')], ignore_index=True)
    df = pd.DataFrame({
        'MedInc':      raw['median_income'],                       # median income (tens of $1,000s)
        'HouseAge':    raw['housing_median_age'],
        'AveRooms':    raw['total_rooms'] / raw['households'],
        'AveBedrms':   raw['total_bedrooms'] / raw['households'],
        'Population':  raw['population'],
        'AveOccup':    raw['population'] / raw['households'],
        'Latitude':    raw['latitude'],
        'Longitude':   raw['longitude'],
        'MedHouseVal': raw['median_house_value'] / 1e5,            # target, $100,000s
    })
    DATA_SOURCE = 'Colab sample_data/california_housing_{train,test}.csv'
else:
    from sklearn.datasets import fetch_california_housing
    df = fetch_california_housing(as_frame=True).frame
    DATA_SOURCE = 'sklearn.datasets.fetch_california_housing'

df = df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
NONSPATIAL = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup']
COORDS     = ['Longitude', 'Latitude']
TARGET     = 'MedHouseVal'
y = df[TARGET].values.astype('float32')
print(f'{len(df):,} block groups from {DATA_SOURCE}')
df.head()

In [ ]:
ASPECT = 1 / np.cos(np.radians(36))      # keeps lon/lat maps roughly true to shape at California's latitude

def lonlat_map(ax, values, title, cmap='viridis', s=2, **kw):
    sc = ax.scatter(df['Longitude'], df['Latitude'], c=values, s=s, cmap=cmap, **kw)
    ax.set_aspect(ASPECT); ax.set_title(title); ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    return sc

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
plt.colorbar(lonlat_map(axes[0], df[TARGET], 'Median house value ($100k)'), ax=axes[0], shrink=0.8)
plt.colorbar(lonlat_map(axes[1], np.log(df['MedInc']), 'log median income', cmap='magma'), ax=axes[1], shrink=0.8)
plt.tight_layout(); plt.show()

**Question 0.** In two sentences: where are the most expensive block groups, and can you *see* Tobler's first law ("near things are more related than distant things") in these maps?

*Your answer:*

## Part 1 · Baselines with and without coordinates

Before any deep learning, set a baseline. We fit a linear regression and a random forest, each with and without longitude and latitude, and score them with ordinary **random** 5-fold cross-validation (CV). Skewed count variables are log-transformed first.

In [ ]:
SKEWED = ['AveRooms', 'AveBedrms', 'Population', 'AveOccup']

def make_X(frame, cols):
    X = frame[cols].copy()
    for c in SKEWED:
        if c in X:
            X[c] = np.log1p(X[c])
    return X.values.astype('float32')

def cv_score(model_fn, X, y, splitter, groups=None):
    # Pooled out-of-fold predictions: every row is predicted by a model that never saw it.
    pred = np.zeros_like(y)
    for tr, te in splitter.split(X, y, groups):
        m = model_fn(); m.fit(X[tr], y[tr]); pred[te] = m.predict(X[te])
    return r2_score(y, pred), np.sqrt(mean_squared_error(y, pred)), pred

MODELS = {
    'Linear':        LinearRegression,
    'Random forest': lambda: RandomForestRegressor(n_estimators=60, min_samples_leaf=3, n_jobs=-1, random_state=SEED),
}
FEATURES = {'no location': NONSPATIAL, '+ coordinates': NONSPATIAL + COORDS}

random_cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows, oof = [], {}
for mname, mfn in MODELS.items():
    for fname, cols in FEATURES.items():
        t0 = time.time()
        r2, rmse, pred = cv_score(mfn, make_X(df, cols), y, random_cv)
        oof[(mname, fname, 'random')] = pred
        cv_rows.append(dict(model=mname, features=fname, cv='random', R2=r2, RMSE=rmse))
        print(f'{mname:14s} {fname:14s} R2={r2:.3f}  ({time.time()-t0:.1f}s)')
pd.DataFrame(cv_rows).round(3)

**Question 1.** How much do coordinates help each model? Why does the random forest gain more from longitude and latitude than linear regression does? (Hint: what shape of surface can a linear model draw over lon/lat?)

*Your answer:*

## Part 2 · Spatial autocorrelation and spatial cross-validation

### 2a · Moran's I
Moran's I measures whether values at nearby locations are more similar than chance. With row-standardised weights it is

$$ I = \frac{\sum_i z_i \, \bar{z}_{N(i)}}{\sum_i z_i^2} $$

where $z$ is the centred variable and $\bar z_{N(i)}$ is the mean of $i$'s $k$ nearest neighbours (its *spatial lag*). $I \approx 0$ means no spatial pattern; values towards 1 mean strong clustering. A permutation test gives a pseudo p-value. In practice you would use [PySAL's `esda`](https://pysal.org/esda/); here it is written out so you can see every step.

In [ ]:
def knn_neighbours(lon, lat, k=8):
    # k nearest neighbours by great-circle (haversine) distance; returns an (n, k) index array without self.
    coords = np.radians(np.c_[lat, lon])
    _, idx = NearestNeighbors(n_neighbors=k + 1, metric='haversine').fit(coords).kneighbors(coords)
    return idx[:, 1:]

def morans_i(values, nbrs, n_perm=199, seed=SEED):
    z = np.asarray(values, float); z = z - z.mean()
    I = (z @ z[nbrs].mean(axis=1)) / (z @ z)
    rng = np.random.default_rng(seed)
    perm = [(zp @ zp[nbrs].mean(axis=1)) / (zp @ zp) for zp in (rng.permutation(z) for _ in range(n_perm))]
    p = (np.sum(np.array(perm) >= I) + 1) / (n_perm + 1)
    return I, p

NBRS = knn_neighbours(df['Longitude'].values, df['Latitude'].values, k=8)
for label, v in [('House value (target)', y),
                 ('Residuals: random forest, no location', y - oof[('Random forest', 'no location', 'random')]),
                 ('Residuals: random forest, + coordinates', y - oof[('Random forest', '+ coordinates', 'random')])]:
    I, p = morans_i(v, NBRS)
    print(f"{label:42s} Moran's I = {I:.3f}  (p = {p:.3f})")

Residuals that are still spatially autocorrelated tell you the model has missed spatial structure. They also warn you about evaluation: if a test block group has a near-twin next door in the training set, random CV rewards the model for *interpolating*, not for generalising to new places.

### 2b · Spatial block cross-validation
We cluster the coordinates into spatial blocks and hold out whole blocks at a time (`GroupKFold`), so test data are geographically separated from training data (Roberts et al., 2017).

In [ ]:
N_BLOCKS = 40
xy = df[COORDS].values * np.array([np.cos(np.radians(36)), 1.0])     # approximate equal-distance scaling
blocks = KMeans(n_clusters=N_BLOCKS, n_init=10, random_state=SEED).fit_predict(xy)
spatial_cv = GroupKFold(n_splits=5)

fold_random  = np.zeros(len(df), int)
fold_spatial = np.zeros(len(df), int)
for f, (_, te) in enumerate(random_cv.split(df)):                fold_random[te] = f
for f, (_, te) in enumerate(spatial_cv.split(df, y, blocks)):    fold_spatial[te] = f

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
lonlat_map(axes[0], fold_random,  'Random 5-fold CV: test folds are scattered', cmap='tab10', vmin=0, vmax=9)
lonlat_map(axes[1], fold_spatial, 'Spatial block CV: test folds are regions',  cmap='tab10', vmin=0, vmax=9)
plt.tight_layout(); plt.show()

In [ ]:
for mname, mfn in MODELS.items():
    for fname, cols in FEATURES.items():
        r2, rmse, pred = cv_score(mfn, make_X(df, cols), y, spatial_cv, groups=blocks)
        oof[(mname, fname, 'spatial')] = pred
        cv_rows.append(dict(model=mname, features=fname, cv='spatial', R2=r2, RMSE=rmse))

cv_table = (pd.DataFrame(cv_rows)
              .pivot_table(index=['model', 'features'], columns='cv', values='R2')
              .assign(drop=lambda t: t['random'] - t['spatial']))
cv_table.round(3)

**Question 2.** (a) Which R² would you report in a paper, and why? (b) Is the drop from random to spatial CV larger with or without coordinates? Explain using the idea of *memorising places*.

**Task 2.** Change `N_BLOCKS` to 10 and then to 200 and re-run the two cells above. How does block size change the gap? What real-world question does each block size correspond to?

*Your answer:*

## Part 3 · Neural networks with three kinds of location input

From here on we use **one spatial split** so that every model is tested on the same unseen regions: spatial fold 0 is the test region, fold 1 is a validation region for early stopping, and the rest is for training.

In [ ]:
test_mask  = fold_spatial == 0
val_mask   = fold_spatial == 1
train_mask = ~(test_mask | val_mask)
split_code = np.where(test_mask, 2, np.where(val_mask, 1, 0))

fig, ax = plt.subplots(figsize=(6.5, 5.5))
lonlat_map(ax, split_code, 'Train (0) / validation (1) / test (2) regions', cmap='viridis', vmin=0, vmax=2)
plt.show()
print(f'train {train_mask.sum():,} | validation {val_mask.sum():,} | test {test_mask.sum():,}')

results = []   # every model from now on is scored on the same test region
def report(name, pred, mask=None):
    mask = test_mask if mask is None else mask
    row = dict(model=name, R2=r2_score(y[mask], pred[mask]), RMSE=float(np.sqrt(mean_squared_error(y[mask], pred[mask]))))
    results.append(row); return row

for fname, cols in FEATURES.items():
    X = make_X(df, cols)
    rf = MODELS['Random forest']().fit(X[train_mask], y[train_mask])
    print(report(f'Random forest, {fname}', rf.predict(X)))

### Three ways to give a network location

1. **No location.** The MLP sees only the six census variables.
2. **Raw coordinates.** Append longitude and latitude as two extra inputs. A ReLU network has to bend these two numbers into a complex surface, which it learns slowly.
3. **Multi-scale sinusoidal encoding.** Replace each coordinate by $\sin(2\pi x/\lambda)$ and $\cos(2\pi x/\lambda)$ at several wavelengths $\lambda$, from a few kilometres to hundreds. This is the idea behind *Space2Vec*'s grid-cell encoder (Mai et al., 2020) and the location encoders reviewed in this week's reading (Mai et al., 2022).

In [ ]:
def raw_coords(frame):
    return frame[COORDS].values.astype('float32')

def sinusoidal_encoding(frame, n_scales=12, min_wavelength=0.05, max_wavelength=10.0):
    # Multi-scale sin/cos features of lon and lat; wavelengths in degrees (0.05 deg ~ 5 km).
    lon = frame['Longitude'].values[:, None]
    lat = frame['Latitude'].values[:, None]
    lam = np.geomspace(min_wavelength, max_wavelength, n_scales)[None, :]
    feats = [np.sin(2 * np.pi * lon / lam), np.cos(2 * np.pi * lon / lam),
             np.sin(2 * np.pi * lat / lam), np.cos(2 * np.pi * lat / lam)]
    return np.hstack(feats).astype('float32')

# What does the encoding look like? Three of its 48 channels, mapped:
enc = sinusoidal_encoding(df)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, ch, label in zip(axes, [0, 6, 11], ['finest', 'middle', 'coarsest']):
    lonlat_map(ax, enc[:, ch], f'sin(lon) channel {ch} ({label} wavelength)', cmap='coolwarm', s=1)
plt.tight_layout(); plt.show()

In [ ]:
class MLP(nn.Module):
    def __init__(self, d_in, hidden=128, depth=3, dropout=0.1):
        super().__init__()
        layers, d = [], d_in
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(dropout)]
            d = hidden
        layers.append(nn.Linear(d, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_mlp(X, epochs=200, patience=20, lr=1e-3, batch=256, hidden=128, depth=3, seed=SEED):
    # Mini-batch training with early stopping on the validation region. Returns predictions for every row.
    set_seed(seed)
    scaler = StandardScaler().fit(X[train_mask])
    Xs = torch.tensor(scaler.transform(X), dtype=torch.float32, device=device)
    yt = torch.tensor(y, device=device)
    tr = torch.tensor(np.where(train_mask)[0], device=device)
    model = MLP(X.shape[1], hidden, depth).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    best, best_state, wait, history = np.inf, None, 0, []
    for epoch in range(epochs):
        model.train()
        perm = tr[torch.randperm(len(tr), device=device)]
        for i in range(0, len(perm), batch):
            b = perm[i:i + batch]
            loss = F.mse_loss(model(Xs[b]), yt[b])
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            pred = model(Xs).cpu().numpy()
        val_rmse = float(np.sqrt(mean_squared_error(y[val_mask], pred[val_mask])))
        history.append(val_rmse)
        if val_rmse < best - 1e-4:
            best, wait = val_rmse, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        return model(Xs).cpu().numpy(), history

In [ ]:
X_base = make_X(df, NONSPATIAL)
MLP_INPUTS = {
    'MLP, no location':         X_base,
    'MLP, raw lon/lat':         np.hstack([X_base, raw_coords(df)]),
    'MLP, sinusoidal encoding': np.hstack([X_base, sinusoidal_encoding(df)]),
}
preds, histories = {}, {}
for name, X in MLP_INPUTS.items():
    t0 = time.time()
    preds[name], histories[name] = train_mlp(X)
    print(report(name, preds[name]), f'| {len(histories[name])} epochs, {time.time()-t0:.0f}s')

plt.figure(figsize=(7, 4))
for name, h in histories.items():
    plt.plot(h, label=name)
plt.xlabel('epoch'); plt.ylabel('validation RMSE ($100k)'); plt.title('Learning curves on the validation region'); plt.legend(); plt.show()

**Question 3.** Rank the three MLPs on the unseen test region. Which *inductive bias* does the sinusoidal encoding add that raw coordinates lack?

**Task 3.** Re-run the encoding with `min_wavelength=0.005` (about 500 m) and then with `min_wavelength=1.0`, keeping everything else fixed. Use the cell below. What happens on the spatial test region, and why might very fine wavelengths *hurt* when the test data are in new places?

In [ ]:
# Task 3: try other wavelengths here
for mw in [0.005, 1.0]:
    X = np.hstack([X_base, sinusoidal_encoding(df, min_wavelength=mw)])
    p, _ = train_mlp(X)
    row = dict(model=f'MLP, sinusoidal (min wavelength {mw} deg)', R2=r2_score(y[test_mask], p[test_mask]))
    print(row)

*Your answer:*

## Part 4 · A convolution is a GIS focal operation

A CNN's basic operation slides a small grid of weights (a *kernel*) over a raster and takes a weighted sum at each position. That is exactly a **focal** (moving-window) operation from map algebra. The only difference is that a CNN *learns* its kernel weights, and stacks many of them.

We first turn median income into a raster on a 0.1° grid.

In [ ]:
RES = 0.1
lon_edges = np.arange(-124.5, -114.0 + RES, RES)
lat_edges = np.arange(32.4, 42.2 + RES, RES)
s, _, _ = np.histogram2d(df['Latitude'], df['Longitude'], bins=[lat_edges, lon_edges], weights=df['MedInc'])
c, _, _ = np.histogram2d(df['Latitude'], df['Longitude'], bins=[lat_edges, lon_edges])
grid = np.where(c > 0, s / np.maximum(c, 1), np.nan).astype('float32')     # mean income per cell; NaN = NoData
filled = np.nan_to_num(grid)
print('raster shape (rows = latitude, cols = longitude):', grid.shape, '| cells with data:', int((c > 0).sum()))

# 1) GIS-style focal mean with SciPy (3 x 3 window, zeros beyond the edge)
gis_focal = ndimage.uniform_filter(filled, size=3, mode='constant', cval=0.0)

# 2) The same thing as a deep-learning convolution
kernel = torch.full((1, 1, 3, 3), 1 / 9)
dl_focal = F.conv2d(torch.tensor(filled)[None, None], kernel, padding=1)[0, 0].numpy()
print('max |GIS focal mean - torch conv2d| =', float(np.abs(gis_focal - dl_focal).max()))

The two answers agree to floating-point precision. Real rasters have NoData, though. A GIS ignores empty cells; with convolutions the trick is **normalised convolution**: convolve the values and the data mask separately, then divide.

In [ ]:
def conv(raster, k):
    k = torch.as_tensor(k, dtype=torch.float32)[None, None]
    return F.conv2d(torch.tensor(raster, dtype=torch.float32)[None, None], k, padding=k.shape[-1] // 2)[0, 0].numpy()

mask = (~np.isnan(grid)).astype('float32')
num, den = conv(filled * mask, np.ones((3, 3))), conv(mask, np.ones((3, 3)))
focal_nodata = np.where(mask > 0, num / np.maximum(den, 1), np.nan)

SOBEL_X = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype='float32')
SOBEL_Y = SOBEL_X.T
edges = np.hypot(conv(filled, SOBEL_X), conv(filled, SOBEL_Y))
edges = np.where(den == 9, edges, np.nan)      # only cells whose whole 3 x 3 window has data: NoData edges are not income edges

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, img, title, cmap in zip(axes, [grid, focal_nodata, edges],
                                ['Median income raster', 'Focal mean (NoData-aware)', 'Sobel edge strength (complete windows only)'],
                                ['magma', 'magma', 'inferno']):
    im = ax.imshow(img, origin='lower', cmap=cmap, extent=[lon_edges[0], lon_edges[-1], lat_edges[0], lat_edges[-1]])
    ax.set_aspect(ASPECT); ax.set_title(title); plt.colorbar(im, ax=ax, shrink=0.75)
plt.tight_layout(); plt.show()

### From fixed to learned kernels
In a CNN nobody hand-picks the Sobel weights. Below, a single 3 × 3 convolution starts from random weights and is trained by gradient descent to reproduce the Sobel-x output. Watch it *discover* the kernel.

In [ ]:
set_seed()
x_in = torch.tensor(filled)[None, None]
target = torch.tensor(conv(filled, SOBEL_X))[None, None]
learner = nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False)
print('random start:\n', learner.weight.data[0, 0].numpy().round(2))
opt = torch.optim.Adam(learner.parameters(), lr=0.05)
for step in range(600):
    loss = F.mse_loss(learner(x_in), target)
    opt.zero_grad(); loss.backward(); opt.step()
print(f'after training (loss {loss.item():.5f}):\n', learner.weight.data[0, 0].numpy().round(2))
print('Sobel-x:\n', SOBEL_X)

**Task 4.** Design a 3 × 3 kernel that highlights cells **richer than their surroundings** (local hot spots), apply it with `conv(filled, your_kernel)` and map the result. Then open the solution.

In [ ]:
# Task 4: your kernel here
my_kernel = np.zeros((3, 3), dtype='float32')

In [ ]:
# @title Solution: a Laplacian-style hot-spot kernel (double-click to show) { display-mode: "form" }
hot = np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]], dtype='float32') / 8
hotspots = np.where(mask > 0, conv(filled, hot), np.nan)
plt.figure(figsize=(6.5, 5))
plt.imshow(hotspots, origin='lower', cmap='RdBu_r', vmin=-3, vmax=3, extent=[lon_edges[0], lon_edges[-1], lat_edges[0], lat_edges[-1]])
plt.gca().set_aspect(ASPECT); plt.colorbar(shrink=0.8); plt.title('Cell minus mean of its 8 neighbours'); plt.show()

## Part 5 · A graph neural network on a neighbourhood graph

Census areas are not pixels on a regular grid, so we connect each block group to its $k$ nearest neighbours and let a **graph neural network** pass messages along those links. In each layer, every node combines its *own* features with the *average of its neighbours'* features (the GraphSAGE form; Hamilton et al., 2017):

$$ h_i^{(l+1)} = \sigma\Big(W^{(l)}\big[\,h_i^{(l)} \;\|\; \tfrac{1}{|N(i)|}\textstyle\sum_{j \in N(i)} h_j^{(l)}\big]\Big) $$

Two layers let information travel two hops. Note that **the network never sees a coordinate**: location enters only through *who is connected to whom*.

In [ ]:
def normalised_adjacency(nbrs, self_loops=True, symmetric=True):
    n, k = nbrs.shape
    rows, cols = np.repeat(np.arange(n), k), nbrs.ravel()
    r, c = np.concatenate([rows, cols]), np.concatenate([cols, rows])          # make the graph undirected
    if self_loops:
        r, c = np.concatenate([r, np.arange(n)]), np.concatenate([c, np.arange(n)])
    idx = torch.tensor(np.vstack([r, c]), dtype=torch.long)
    A = torch.sparse_coo_tensor(idx, torch.ones(idx.shape[1]), (n, n)).coalesce()
    A = torch.sparse_coo_tensor(A.indices(), torch.ones(A.indices().shape[1]), (n, n)).coalesce()   # duplicates -> 1
    deg = torch.sparse.sum(A, dim=1).to_dense()
    i, j = A.indices()
    w = deg[i].pow(-0.5) * deg[j].pow(-0.5) if symmetric else 1.0 / deg[i]
    return torch.sparse_coo_tensor(A.indices(), w, (n, n)).coalesce().to(device)

K_GRAPH = 8
GRAPH_NBRS = knn_neighbours(df['Longitude'].values, df['Latitude'].values, k=K_GRAPH)
A_mean = normalised_adjacency(GRAPH_NBRS, self_loops=False, symmetric=False)   # row i averages i's neighbours
print(f'{A_mean.shape[0]:,} nodes, {A_mean._nnz():,} directed edges after symmetrising')

In [ ]:
class GraphSAGE(nn.Module):
    def __init__(self, d_in, hidden=64, dropout=0.1):
        super().__init__()
        self.l1, self.l2, self.out = nn.Linear(2 * d_in, hidden), nn.Linear(2 * hidden, hidden), nn.Linear(hidden, 1)
        self.dropout = dropout
    def forward(self, x, A_mean):
        h = F.relu(self.l1(torch.cat([x, torch.sparse.mm(A_mean, x)], dim=1)))   # layer 1: me || mean of my neighbours
        h = F.dropout(h, self.dropout, self.training)
        h = F.relu(self.l2(torch.cat([h, torch.sparse.mm(A_mean, h)], dim=1)))   # layer 2: information now reaches 2 hops
        return self.out(h).squeeze(-1)

def train_graph_model(model, X, A, epochs=1500, eval_every=5, patience=30, lr=1e-2, seed=SEED):
    # Full-batch training: one gradient step per epoch on the whole graph. Every node's FEATURES are used,
    # but the loss only sees TARGETS in the training region. Early stopping on the validation region.
    set_seed(seed)
    scaler = StandardScaler().fit(X[train_mask])
    Xs = torch.tensor(scaler.transform(X), dtype=torch.float32, device=device)
    yt = torch.tensor(y, device=device)
    tr = torch.tensor(train_mask, device=device)
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    best, best_state, wait, history = np.inf, None, 0, []
    for epoch in range(epochs):
        model.train()
        loss = F.mse_loss(model(Xs, A)[tr], yt[tr])
        opt.zero_grad(); loss.backward(); opt.step()
        if epoch % eval_every:
            continue
        model.eval()
        with torch.no_grad():
            pred = model(Xs, A).cpu().numpy()
        val_rmse = float(np.sqrt(mean_squared_error(y[val_mask], pred[val_mask])))
        history.append(val_rmse)
        if val_rmse < best - 1e-4:
            best, wait = val_rmse, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        return model(Xs, A).cpu().numpy(), history

t0 = time.time()
name = 'GNN (GraphSAGE-style), graph only'
preds[name], histories[name] = train_graph_model(GraphSAGE(X_base.shape[1]), X_base, A_mean)
print(report(name, preds[name]), f'| about {len(histories[name]) * 5} epochs, {time.time()-t0:.0f}s')

**Question 5.** (a) Compare the GNN with *MLP, no location*: both see the same six variables, so what explains the difference? (b) Test-region nodes are in the graph, so their **features** (never their house values) flow to neighbours during training. Is that fair? When would it be realistic, and when would it be leakage?

**Task 5.** The classic **graph convolutional network** (GCN; Kipf and Welling, 2017) does not keep a node's own features apart: it averages each node *together with* its neighbours, using a symmetric normalisation,

$$ H^{(l+1)} = \sigma\big(\hat{A}\, H^{(l)} W^{(l)}\big), \qquad \hat{A} = D^{-1/2}(A + I)\,D^{-1/2}. $$

Implement a two-layer GCN below (build $\hat A$ with `normalised_adjacency(GRAPH_NBRS)`), train it with `train_graph_model`, and compare it with the GraphSAGE-style model. If it does worse, why? (Hint: what happens to a node's own median income?) Then try `K_GRAPH = 4` and `32`.

In [ ]:
# Task 5: implement a classic GCN here
A_hat = normalised_adjacency(GRAPH_NBRS)          # D^-1/2 (A + I) D^-1/2

class GCN(nn.Module):
    def __init__(self, d_in, hidden=64, dropout=0.1):
        super().__init__()
        ...  # your layers
    def forward(self, x, A_hat):
        ...  # your forward pass

In [ ]:
# @title Solution: a two-layer GCN (double-click to show) { display-mode: "form" }
class GCN(nn.Module):
    def __init__(self, d_in, hidden=64, dropout=0.1):
        super().__init__()
        self.lin1, self.lin2, self.out = nn.Linear(d_in, hidden), nn.Linear(hidden, hidden), nn.Linear(hidden, 1)
        self.dropout = dropout
    def forward(self, x, A_hat):
        h = F.relu(torch.sparse.mm(A_hat, self.lin1(x)))     # transform, then average over node + neighbours
        h = F.dropout(h, self.dropout, self.training)
        h = F.relu(torch.sparse.mm(A_hat, self.lin2(h)))
        return self.out(h).squeeze(-1)

A_hat = normalised_adjacency(GRAPH_NBRS)
name = 'GCN (Kipf & Welling), graph only'
preds[name], histories[name] = train_graph_model(GCN(X_base.shape[1]), X_base, A_hat)
print(report(name, preds[name]))

## Part 6 · Compare, reflect and save

In [ ]:
res = (pd.DataFrame(results).drop_duplicates('model', keep='last')
         .sort_values('R2').reset_index(drop=True))
fig, ax = plt.subplots(figsize=(8, 0.45 * len(res) + 1))
ax.barh(res['model'], res['R2'], color=['#4C72B0' if 'GNN' in m or 'GCN' in m else ('#DD8452' if 'MLP' in m else '#8C8C8C') for m in res['model']])
for i, v in enumerate(res['R2']):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center')
ax.set_xlabel('R² on the unseen test region (higher is better)'); ax.set_xlim(min(0, res['R2'].min() - 0.05), max(1.0, res['R2'].max() + 0.08))
ax.set_title('Same data, different ways of adding space'); plt.tight_layout(); plt.show()
res.round(3)

In [ ]:
# Where does the best model still fail? Residuals in the test region, and their spatial autocorrelation.
best_name = max(preds, key=lambda k: r2_score(y[test_mask], preds[k][test_mask]))
resid = y - preds[best_name]
test_idx = np.where(test_mask)[0]
nb_test = knn_neighbours(df['Longitude'].values[test_idx], df['Latitude'].values[test_idx], k=8)
for name, p in preds.items():
    I, pval = morans_i((y - p)[test_idx], nb_test, n_perm=99)
    print(f"{name:40s} test-region residual Moran's I = {I:.3f}")

fig, ax = plt.subplots(figsize=(7, 5.5))
sc = ax.scatter(df['Longitude'][test_mask], df['Latitude'][test_mask], c=resid[test_mask], s=4, cmap='RdBu_r', vmin=-1.5, vmax=1.5)
ax.set_aspect(ASPECT); plt.colorbar(sc, label='observed - predicted ($100k)'); ax.set_title(f'Residuals of best model: {best_name}')
plt.show()

### Reflection (answers feed next week's opening poll)

1. Which way of adding space helped most on the unseen region, and which inductive bias explains it?
2. Your random-CV and spatial-CV scores differ. Which would you report, and what question does each one answer?
3. Next week we classify Sentinel-2 tiles with CNNs. Using Part 4, explain in one sentence what the first layer of a CNN does to an image.
4. **Project seed.** Name one dataset you might use for your course project, and the architecture from today that suits its structure (grid, graph, sequence or points).

*Your answers:*

### Save your work reproducibly
Run the cell below, then use **File → Save a copy in GitHub** to commit this notebook to *your own* lab repository. You will build your project repository the same way.

In [ ]:
res.to_csv('week1_results.csv', index=False)
with open('week1_environment.txt', 'w') as f:
    f.write(VERSIONS + f'\nseed={SEED}\ndata={DATA_SOURCE}\nN_BLOCKS={N_BLOCKS}\nK_GRAPH={K_GRAPH}\n')
print(open('week1_environment.txt').read())

## Stretch tasks (optional)

**A · Pretrained location embeddings (SatCLIP).** SatCLIP (Klemmer et al., 2025) learns a global location encoder by matching coordinates to Sentinel-2 images. Replace the sinusoidal encoding with SatCLIP embeddings and compare on the test region. The loading code below follows the [SatCLIP README](https://github.com/microsoft/satclip); coordinates go in as a float64 tensor in **(lon, lat)** order. If an import fails, install what the repository's requirements list, or follow its Colab notebook *A01*.

```python
!git clone https://github.com/microsoft/satclip.git
import sys; sys.path.append('./satclip/satclip')
from huggingface_hub import hf_hub_download
from load import get_satclip
satclip = get_satclip(hf_hub_download('microsoft/SatCLIP-ResNet18-L10', 'satclip-resnet18-l10.ckpt'), device=device)
satclip.eval()
with torch.no_grad():
    coords = torch.tensor(df[['Longitude', 'Latitude']].values).double().to(device)
    emb = satclip(coords).float().cpu().numpy()
preds['MLP, SatCLIP'], _ = train_mlp(np.hstack([X_base, emb]))
print(report('MLP, SatCLIP embedding', preds['MLP, SatCLIP']))
```

**B · Attention on graphs.** Install PyTorch Geometric (`pip install torch_geometric`) and replace the GraphSAGE-style model with a graph attention network (`GATConv`). Do the learned attention weights favour some neighbours?

**C · Buffered spatial CV.** Remove training points within 10 km of each test block before fitting. How much further does the score fall?

## References
- Hamilton, W.L., Ying, R. and Leskovec, J. (2017) Inductive representation learning on large graphs. *NeurIPS* 30. [arXiv:1706.02216](https://arxiv.org/abs/1706.02216)
- Kipf, T.N. and Welling, M. (2017) Semi-supervised classification with graph convolutional networks. *ICLR*. [arXiv:1609.02907](https://arxiv.org/abs/1609.02907)
- Klemmer, K., Rolf, E., Robinson, C., Mackey, L. and Rußwurm, M. (2025) SatCLIP: global, general-purpose location embeddings with satellite imagery. *AAAI* 39(4), 4347–4355. [doi:10.1609/aaai.v39i4.32457](https://doi.org/10.1609/aaai.v39i4.32457)
- Mai, G., Janowicz, K., Yan, B., Zhu, R., Cai, L. and Lao, N. (2020) Multi-scale representation learning for spatial feature distributions using grid cells. *ICLR*. [arXiv:2003.00824](https://arxiv.org/abs/2003.00824)
- Mai, G. et al. (2022) A review of location encoding for GeoAI: methods and applications. *IJGIS* 36(4), 639–673. [doi:10.1080/13658816.2021.2004602](https://doi.org/10.1080/13658816.2021.2004602)
- Pace, R.K. and Barry, R. (1997) Sparse spatial autoregressions. *Statistics & Probability Letters* 33(3), 291–297.
- Roberts, D.R. et al. (2017) Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. *Ecography* 40(8), 913–929. [doi:10.1111/ecog.02881](https://doi.org/10.1111/ecog.02881)